In [1]:
from typing import Dict, List, TypedDict, Any, cast
from langgraph.graph import StateGraph, END

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.embeddings import SentenceTransformerEmbeddings

In [2]:
import dotenv

dotenv.load_dotenv()

True

In [3]:
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


In [4]:
from langfuse.langchain import CallbackHandler


langfuse_handler = CallbackHandler()

In [5]:
import numpy as np
import json
import sqlite3
import sqlite_vec
from kiwipiepy import Kiwi


DB_PATH = "/home/codeitDev/project/AI_7-team/DB/document.db"
MODEL_NAME = "jhgan/ko-sroberta-multitask"

embeddings = SentenceTransformerEmbeddings(model_name=MODEL_NAME)

kiwi = Kiwi()

class SearchState(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    # 추후, 전처리 단계 개선을 수행한 뒤에 적용 예정
    # scopes: List[Dict]

    # 검색 결과
    dense_result: List[Dict]
    sparse_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def extract_nouns(query):
    if not query:
        return ""
    tokens: List[Any] = cast(List[Any], kiwi.tokenize(query))
    nouns = [f'"{t.form}"' for t in tokens if t.tag in ('NNG', 'NNP', 'NNB')]
    if len(nouns) == 0:
        return ""
    return " OR ".join(nouns)


def dense_search(state: SearchState):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def sparse_search(state: SearchState):
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ?
                ORDER BY score
                LIMIT 30
        """, (query,))

        sparse_result = cursor.fetchall()
        uids = [r[0] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    return {'sparse_result': final_results}


def rrf(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'])
    sparse_scores = compute_scores(state['sparse_result'])

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:10]]
    return {'search_result': result}

/tmp/ipykernel_43344/841832845.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name=MODEL_NAME)


In [6]:
def empty(SearchState):
    return

In [7]:
search_workflow = StateGraph(SearchState)

search_workflow.add_node("empty", empty)

search_workflow.add_node("dense", dense_search)
search_workflow.add_node("sparse", sparse_search)

search_workflow.add_node("rrf", rrf)

search_workflow.set_entry_point("empty")

search_workflow.add_edge("empty", "dense")
search_workflow.add_edge("empty", "sparse")

search_workflow.add_edge("dense", "rrf")
search_workflow.add_edge("sparse", "rrf")
search_workflow.add_edge("rrf", END)

In [8]:
hybrid_app = search_workflow.compile()

In [9]:
from langchain_core.output_parsers import JsonOutputParser

class RAGState(TypedDict):
    query: str
    context: list[str]
    last_search_query: str | None
    search_result: list[str] | None
    can_answer: bool
    next_query: str
    iteration: int

    final_answer: str

In [10]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

prompt1 = ChatPromptTemplate.from_messages([
    ("system", """
너는 RAG 시스템의 판단 노드다.

역할:
1. 현재 context만으로 original_query에 답할 수 있는지 판단하라.
2. 충분하거나, iteration이 6이면 can_answer=true로 설정하라.
3. 부족하다면 can_answer=false로 설정하고
   last_query가 빈 문자열이 아닌 경우 참조하여
   답을 얻기 위해 필요한 검색 질의(search_query)를 하나 생성하라.

주의:
- 절대 최종 답변을 생성하지 마라.
- reasoning을 출력하지 마라.
- 반드시 다음과 같은 형식의 JSON 형식으로만 출력하라.
    {{
        "can_answer": bool,
        "search_query": str | None
    }}
"""),
    ("human", """
[Original Query]
{original_query}

[Current Context]
{context}
     
[Last Query]
{last_query}

Iteration: {iteration}
""")
])

define_chain = prompt1 | llm | JsonOutputParser()
    
def llm1_node(state: Dict[str, Any]) -> Dict[str, Any]:    
    if 'next_query' in state:
        last_search_query = state['next_query']
    else:
        last_search_query = ''

    result = define_chain.invoke({
        "original_query": state["query"],
        "context": state["context"],
        "iteration": state["iteration"],
        'last_query': last_search_query
    })

    return {
        "can_answer": result["can_answer"],
        "last_search_query": last_search_query,
        "next_query": result["search_query"],
        "iteration": state["iteration"] + 1
    }

In [11]:
def router(state: RAGState):
    if state['can_answer']:
        return 'final_answer'
    return 'next_search'

In [18]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", """
너는 RAG 시스템의 context 결정 노드다.

역할:
1. search_result는 여러 context로 구성되는 배열이다. 각 context는 'text', 'metadata'로 구성되는 배열이다.
    original_query에 대한 답변에 직접적으로 기여하는 context만 선택하라.
    search_query는 original_query에 답하기 위해 생성된 검색 쿼리이므로, 참고만 하라.
    - 먼저 나오는 context부터 순서대로 추가 여부를 결정한다.
    - 새로운 정보를 제공하지 않는 중복 context는 선택하지 않는다.
    - 단순히 동일 키워드를 포함하는 것만으로는 선택하지 않는다.
2. 추가할 context들의 index를 반환하라.

주의:
- 절대 최종 답변을 생성하지 마라.
- reasoning을 출력하지 마라.
- 반드시 다음과 같은 형식의 JSON 형식으로만 출력하라.
    {{
        "indices": list[int]
    }}
"""),
    ("human", """
[Original Query]
{original_query}

[Search Query]
{search_query}

[Search Result]
{search_result}
""")
])

compress_chain = prompt2 | llm | JsonOutputParser()

def context_appender(state: RAGState):

    result = compress_chain.invoke({
        'original_query': state['query'],
        'search_query': state['next_query'],
        'search_result': state['search_result']
    })

    new_items = [state['search_result'][i] for i in result['indices']]
    context = list(state['context']) + new_items
    return {
        'context': context,
        'search_result': []
    }

In [19]:
prompt3 = ChatPromptTemplate.from_messages([
    ("system", """
너는 RAG 시스템의 최종 답변 노드다.

역할:
1. context를 활용하여, original_query에 대한 답변을 구상하라.
2. context 내부의 정보만으로는 답변에 필요한 정보가 충분하지 않다면 반드시 final_answer="답변을 위한 정보가 부족합니다."로 설정한다.

주의:
- reasoning을 출력하지 마라.
- 반드시 context에 있는 정보만을 활용하라.
- 반드시 다음 형식의 string으로 값을 출력하라.
    
    질문 주신 내용에 대한 답은 {{final_answer}}입니다.
"""),
    ("human", """
[Search Query]
{search_query}

[Context]
{context}
""")
])

final_chain = prompt3 | llm | StrOutputParser()


def final_answer(state: RAGState):
    result = final_chain.invoke({
        'search_query': state['query'],
        'context': state['context']
    })
    output = ""
    for i, c in enumerate(state['context']):
        meta = c[1]
        output += f"\n\t{i+1}. {meta['document_title']} 문서의 {meta['page_start']} 페이지부터 {meta['page_end']} 페이지 사이"
    output += "\n에 있습니다."
    result = result + output
    return {'final_answer': result}

In [20]:
builder = StateGraph(RAGState)

builder.add_node("judge", llm1_node)
builder.add_node('retriever', hybrid_app)
builder.add_node('context_appender', context_appender)
builder.add_node('final_answer', final_answer)

builder.set_entry_point('judge')
builder.add_conditional_edges('judge', router,
                            {
                                'final_answer': 'final_answer',
                                'next_search': 'retriever'
                            })
builder.add_edge('retriever', 'context_appender')
builder.add_edge('context_appender', 'judge')
builder.set_finish_point('final_answer')

app = builder.compile()

result = app.invoke({
    'query': '고려대학교에서 발주한 프로젝트의 이름',
    'context': [],
    'iteration': 0
}, config={"callbacks": [langfuse_handler]})

In [21]:
result

{'query': '고려대학교에서 발주한 프로젝트의 이름',
 'context': [('# 제안요청서\n\n고려대학교\n· 차세대 포털 학사 정보시스템 구축 사업\n',
   {'document_title': '차세대 포털 학사 정보시스템 구축사업 -- 고려대학교',
    'source_file': '차세대 포털 학사 정보시스템 구축사업 -- 고려대학교.pdf',
    'section_level1': '제안요청서',
    'section_level2': 'N/A',
    'page_start': 1,
    'page_end': 1,
    'chunk_size': 39,
    'created_at': '2026-02-17T14:20:47.313673',
    'uid': '184cce99e5353df5_1',
    'doc_id': '184cce99e5353df5',
    'type': 'chunk',
    'chunk_order': 1,
    'section_uid': '184cce99e5353df5_h_1'})],
 'last_search_query': None,
 'search_result': [],
 'can_answer': True,
 'next_query': None,
 'iteration': 2,
 'final_answer': '질문 주신 내용에 대한 답은 차세대 포털 학사 정보시스템 구축 사업입니다.\n\t1. 차세대 포털 학사 정보시스템 구축사업 -- 고려대학교 문서의 1 페이지부터 1 페이지 사이\n에 있습니다.'}

In [22]:
result2 = app.invoke({
    'query': '한국 수자원공사에서 발주한 프로젝트의 이름',
    'context': [],
    'iteration': 0
}, config={"callbacks": [langfuse_handler]})
result2

{'query': '한국 수자원공사에서 발주한 프로젝트의 이름',
 'context': [('1.9.8 관계기관협의등\n(1) · K-water 계약상대자는주민 이해당사자의견수렴및관계기관협의등 가시행하는\n, K-water 업무를충실히지원하여야하며 의요구에의해필요한경우업무를\n.\n대행하여야한다\n(2) , , ,\n계약상대자는주요시설물의규모 용수배분량 지자체상수도설치현황및계획\n, ․ , 관로노선및시공한계 인허가등과관련 과업수행기간중관련지자체등과협의가\n․ 필요한사항에대해서는협의를실시하고그결과를검토하여적정방안을수립\n.\n제시하여야한다\n1.9.9 (cts.kwater.or.kr) 계약자통합정보시스템 활용\nK-water 계약자통합정보시스템은계약상대자가용역을수행함에있어 보유시스템을\n활용하여발주자에게전자적문서및정보를제출토록하여계약상대자와발주자간의정보\n( “ CTS” ), 교환및활용을목적으로하며 일명수자원 라고지칭한다\n종이문서를최소화하여\nCALS .\n용역을지원함으로써건설 를구현하기위한것이다\n(1)',
   {'document_title': '용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역 -- 한국수자원공사',
    'source_file': '용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역 -- 한국수자원공사.pdf',
    'section_level1': '목차',
    'section_level2': '1. 일반과업지시서',
    'page_start': 4,
    'page_end': 4,
    'chunk_size': 443,
    'created_at': '2026-02-17T14:14:03.569900',
    'uid': '644031f9a80b79ae_22',
    'doc_id': '644031f9a80b79ae',
    'type': 'chunk',
    'chunk_order': 22,
    'section_uid': '644031f9a80b79ae